In [1]:
import requests
import json
import pandas as pd
from datetime import date
pd.set_option('display.max_rows', None)

team_id = 112
url = "https://statsapi.mlb.com/api/v1/schedule/games/?sportId=1"
team_queried = f"https://statsapi.mlb.com/api/v1/teams/{team_id}"
league = "https://statsapi.mlb.com/api/v1/teams/?sportId=1"

response = requests.get(team_queried)
data = response.json()
team_info = data['teams'][0]

all_info = requests.get(league)
all_data = all_info.json()
all_teams_info = []
for i in range(30):
    all_teams_info.append(all_data['teams'][i])
    #print(all_teams_info[i])

all_team_names = []
for i in range(30):
    all_team_names.append(all_data['teams'][i]['name'])
    #print(all_team_names[i])



for key, value in team_info.items():
    if(key == "teamName"):
        print(f"{key}: {value}")



today = date.today()
schedule_url = f"https://statsapi.mlb.com/api/v1/schedule?teamId={team_id}&startDate=2025-03-10&endDate=2025-10-10&sportId=1"
schedule_response = requests.get(schedule_url)
schedule_data = schedule_response.json()

# Parse schedule data into a DataFrame
games = []
opponent_records = []

opponent_results = {}
dates = schedule_data.get('dates', [])
for day in dates:
    for game in day.get('games', []): 
        opponent = game.get('teams', {}).get('away' if game['teams']['home']['team']['id'] == team_id else 'home', {}).get('team', {}).get('name')
        winner = game['teams']['home'].get('isWinner') if game['teams']['home']['team']['id'] == team_id else game['teams']['away'].get('isWinner')
        wins = game['teams']['home']['leagueRecord'].get('wins') if game['teams']['home']['team']['id'] == team_id else game['teams']['away']['leagueRecord'].get('wins')
        losses = game['teams']['home']['leagueRecord'].get('losses') if game['teams']['home']['team']['id'] == team_id else game['teams']['away']['leagueRecord'].get('losses')
        team_percent = game['teams']['home']['leagueRecord'].get('pct') if game['teams']['home']['team']['id'] == team_id else game['teams']['away']['leagueRecord'].get('pct')
        opponent_percent = game['teams']['away']['leagueRecord'].get('pct') if game['teams']['home']['team']['id'] == team_id else game['teams']['home']['leagueRecord'].get('pct')


        if game['gameType'] == "R": #and game.get('status', {}).get('detailedState') == "Final":
            #if opponent == 'Milwaukee Brewers':
            games.append({
                'Date': game.get('officialDate'),
                'Opponent': opponent,
                'Home/Away': 'Home' if game['teams']['home']['team']['id'] == team_id else 'Away',
                'Status': game.get('status', {}).get('detailedState'),
                'Winner': winner,
                'Record': f"{wins}-{losses}",
                'Percent': team_percent
            })
            opponent_records.append({
                'Opponent': opponent,
                'Opponent Record': opponent_percent
            })

            if game['seriesDescription'] == "Regular Season":
                if opponent not in opponent_results:
                    opponent_results[opponent] = {"Team": opponent, "W": 0, "L": 0, "Future": 0}
                if winner == True:
                    opponent_results[opponent]["W"] += 1
                elif winner == False: 
                    opponent_results[opponent]["L"] += 1
                gameState = game.get('status', {}).get('abstractGameState')
                if (gameState != "Final") and (gameState != "Postponed"):
                    opponent_results[opponent]["Future"] += 1
    

    vs_records = list(opponent_results.values())
    

# 3x + 1.5a + 1.4b + 1.3c + 1.2d + 1.1e
# x = team record and a-e are the last 5 games
# note - maybe multiply last 5 games by opposing team's record to make better opponents worth more to a team's recent strength
# potentially replace a-e with opponent's record from game day
# cubs would be 3(.583) + 1.5(-1) + 1.4(1) + 1.3(1) + 1.2(-1) + 1.1(1) as of 4/20 WITHOUT opponent weighting
# cubs would be 3(.583) - 1.5(.591) + 1.4(.571) + 1.3(.600) - 1.2(.789) + 1.1(.778) as of 4/20 WITH opponent weighting


# Display the schedule
if games:
    df = pd.DataFrame(games)
    df2 = pd.DataFrame(vs_records)
    df3 = pd.DataFrame(opponent_records)
    df2_sorted = df2.sort_values(by='Team')
    print("Team Schedule:")
    print(df)
    print('\n')
    print("Records Against Teams:")
    print(df2_sorted)
    print('\n')
    print("Opponent's Records")
    print(df3)
    #for result in opponent_results:
        #print(opponent_results[result]["Future"])
else:
    print("No scheduled games")



teamName: Cubs
Team Schedule:
           Date               Opponent Home/Away     Status Winner Record  \
0    2025-03-18    Los Angeles Dodgers      Home      Final  False    0-1   
1    2025-03-19    Los Angeles Dodgers      Home      Final  False    0-2   
2    2025-03-27   Arizona Diamondbacks      Away      Final   True    1-2   
3    2025-03-28   Arizona Diamondbacks      Away      Final  False    1-3   
4    2025-03-29   Arizona Diamondbacks      Away      Final   True    2-3   
5    2025-03-30   Arizona Diamondbacks      Away      Final  False    2-4   
6    2025-03-31              Athletics      Away      Final   True    3-4   
7    2025-04-01              Athletics      Away      Final   True    4-4   
8    2025-04-02              Athletics      Away      Final   True    5-4   
9    2025-04-04       San Diego Padres      Home      Final   True    6-4   
10   2025-04-05       San Diego Padres      Home      Final   True    7-4   
11   2025-04-06       San Diego Padres      Ho

In [2]:
last_five = list(games)
last_five.copy()
print(len(opponent_records))
temp_count = (len(last_five)- 5)
print(len(last_five))

for i in range(temp_count):
    del opponent_records[0]
    del last_five[0]
last_five.reverse()
opponent_records.reverse()

#for i in range (len(last_five)):
#    opponent_records.append({
#            'Opponent Percent': opponent_records[i]
#    })


for i in range(len(last_five)):
    print(f"Game {i}: {last_five[i]}")
    print(f"Opponent Record at Game {i+1}: {opponent_records[i]}")

164
164
Game 0: {'Date': '2025-09-28', 'Opponent': 'St. Louis Cardinals', 'Home/Away': 'Home', 'Status': 'Final', 'Winner': True, 'Record': '92-70', 'Percent': '.568'}
Opponent Record at Game 1: {'Opponent': 'St. Louis Cardinals', 'Opponent Record': '.481'}
Game 1: {'Date': '2025-09-27', 'Opponent': 'St. Louis Cardinals', 'Home/Away': 'Home', 'Status': 'Final', 'Winner': True, 'Record': '91-70', 'Percent': '.565'}
Opponent Record at Game 2: {'Opponent': 'St. Louis Cardinals', 'Opponent Record': '.484'}
Game 2: {'Date': '2025-09-26', 'Opponent': 'St. Louis Cardinals', 'Home/Away': 'Home', 'Status': 'Final', 'Winner': True, 'Record': '90-70', 'Percent': '.563'}
Opponent Record at Game 3: {'Opponent': 'St. Louis Cardinals', 'Opponent Record': '.488'}
Game 3: {'Date': '2025-09-25', 'Opponent': 'New York Mets', 'Home/Away': 'Home', 'Status': 'Final', 'Winner': False, 'Record': '89-70', 'Percent': '.560'}
Opponent Record at Game 4: {'Opponent': 'New York Mets', 'Opponent Record': '.516'}
Gam

In [3]:
from datetime import datetime

cubs_id = 112  # Cubs team ID
current_date = datetime.now()

past_home_wins = []
future_home_games = []

for date_entry in schedule_data["dates"]:
    for game in date_entry["games"]:
        game_date = datetime.fromisoformat(game["gameDate"].replace("Z", "+00:00"))
        home_team = game["teams"]["home"]["team"]["id"]
        away_team = game["teams"]["away"]["team"]["id"]
        home_score = game["teams"]["home"].get("score", 0)
        away_score = game["teams"]["away"].get("score", 0)

        is_cubs_home = home_team == cubs_id
        is_cubs_away = away_team == cubs_id

        if is_cubs_home:
            current_date = datetime.now().replace(tzinfo=None)
            game_date = game_date.replace(tzinfo=None)
            if game_date < current_date:
                # Past game
                if is_cubs_home and home_score >= 1:
                    past_home_wins.append({**game, "isCubsHome": True})
                elif is_cubs_away and away_score >= 1:
                    past_home_wins.append({**game, "isCubsHome": False})
            else:
                # Future game
                future_home_games.append(game)

print(future_home_games)

[]


In [4]:
from datetime import date

print({})

{}
